# Feature Engineering V3 — EDA
Exploratory analysis of all `_v2` tables produced by `FeatureEngineering_v3.ipynb`.

**Tables covered:**
- `activity_base_v2`, `activity_labeled_v2`, `activity_dictionary_v2`, `activity_effort_mapping_ai_v2`
- `contact_one_row_v2`, `developer_universe_v2`
- `dev_recency_features_v2`, `dev_features_lifetime_v2`
- `dev_features_0_30d_v2`, `dev_features_30_90d_v2`, `dev_features_90_180d_v2`
- `dev_weekly_features_v2`, `dev_meaningful_week_v2`
- `dev_effort_level_v2`, `dev_persona_v2`, `dev_contact_persona_v2`
- `dev_journey_state_v2`, `dev_dormancy_status_v2`, `dev_activation_v2`
- `dev_profile_final_v4`

## 1. Setup

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)
print("Connected:", DB_PATH)

## 2. Table Inventory

In [ ]:
all_tables = con.execute("SHOW TABLES").fetchdf()
v2_tables = all_tables[all_tables['name'].str.endswith('_v2') | all_tables['name'].str.endswith('_v4')].reset_index(drop=True)

row_counts = []
for t in v2_tables['name']:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    row_counts.append(n)

v2_tables['row_count'] = row_counts
display(v2_tables)

## 3. Activity Data (`activity_labeled_v2`)

In [ ]:
display(con.execute("DESCRIBE activity_labeled_v2").fetchdf())

In [ ]:
cols = [r[0] for r in con.execute("DESCRIBE activity_labeled_v2").fetchall()]
total = con.execute("SELECT COUNT(*) FROM activity_labeled_v2").fetchone()[0]
null_rates = {}
for col in cols:
    n_null = con.execute(f"SELECT COUNT(*) FROM activity_labeled_v2 WHERE {col} IS NULL").fetchone()[0]
    null_rates[col] = round(100 * n_null / total, 2)

null_df = pd.DataFrame.from_dict(null_rates, orient='index', columns=['null_pct']).sort_values('null_pct', ascending=False)
display(null_df[null_df['null_pct'] > 0])

In [ ]:
act_dist = con.execute("""
    SELECT activity, COUNT(*) AS n
    FROM activity_labeled_v2
    GROUP BY activity
    ORDER BY n DESC
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(act_dist['activity'], act_dist['n'])
ax.set_xlabel('Row count')
ax.set_title('Activity type distribution (activity_labeled_v2)')
plt.tight_layout()
plt.show()
display(act_dist)

In [ ]:
timeline = con.execute("""
    SELECT DATE_TRUNC('month', activity_date) AS month, COUNT(*) AS n
    FROM activity_labeled_v2
    WHERE activity_date IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").fetchdf()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(timeline['month'], timeline['n'])
ax.set_title('Monthly activity volume')
ax.set_xlabel('Month')
ax.set_ylabel('Events')
plt.tight_layout()
plt.show()

In [ ]:
jl = con.execute("""
    SELECT journey_label, COUNT(*) AS n
    FROM activity_labeled_v2
    GROUP BY journey_label ORDER BY n DESC
""").fetchdf()
display(jl)

## 4. Effort Mapping (`activity_effort_mapping_ai_v2`)

In [ ]:
display(con.execute("DESCRIBE activity_effort_mapping_ai_v2").fetchdf())

effort = con.execute("""
    SELECT effort_tier, COUNT(*) AS n
    FROM activity_effort_mapping_ai_v2
    GROUP BY effort_tier ORDER BY n DESC
""").fetchdf()
display(effort)

In [ ]:
effort_scores = con.execute("""
    SELECT effort_score
    FROM activity_effort_mapping_ai_v2
    WHERE effort_score IS NOT NULL
""").fetchdf()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(effort_scores['effort_score'], bins=30, edgecolor='white')
ax.set_title('Effort score distribution (activity_effort_mapping_ai_v2)')
ax.set_xlabel('Effort score')
plt.tight_layout()
plt.show()

## 5. Developer Universe (`developer_universe_v2`)

In [ ]:
display(con.execute("DESCRIBE developer_universe_v2").fetchdf())

display(con.execute("""
    SELECT
        COUNT(*) AS total_developers,
        COUNT(CASE WHEN has_contact THEN 1 END) AS with_contact,
        COUNT(CASE WHEN has_activity THEN 1 END) AS with_activity
    FROM developer_universe_v2
""").fetchdf())

## 6. Recency Features (`dev_recency_features_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_recency_features_v2").fetchdf())

In [ ]:
recency_cols = ['days_since_last_activity', 'days_since_first_activity', 'active_days_total']
existing = [r[0] for r in con.execute("DESCRIBE dev_recency_features_v2").fetchall()]
recency_cols = [c for c in recency_cols if c in existing]

if recency_cols:
    stats = con.execute(f"""
        SELECT {', '.join([f'MIN({c}) AS {c}_min, MAX({c}) AS {c}_max, AVG({c}) AS {c}_mean, MEDIAN({c}) AS {c}_median' for c in recency_cols])}
        FROM dev_recency_features_v2
    """).fetchdf()
    display(stats)

In [ ]:
flag_cols = ['has_activity_0_30d', 'has_activity_30_90d', 'has_activity_90_180d', 'newly_inactive_0_30d']
existing = [r[0] for r in con.execute("DESCRIBE dev_recency_features_v2").fetchall()]
flag_cols = [c for c in flag_cols if c in existing]

if flag_cols:
    flag_df = con.execute(f"""
        SELECT {', '.join([f'AVG(CAST({c} AS DOUBLE)) AS pct_{c}' for c in flag_cols])}
        FROM dev_recency_features_v2
    """).fetchdf()
    display(flag_df)

## 7. Lifetime Features (`dev_features_lifetime_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_features_lifetime_v2").fetchdf())

In [ ]:
cols = [r[0] for r in con.execute("DESCRIBE dev_features_lifetime_v2").fetchall()]
total = con.execute("SELECT COUNT(*) FROM dev_features_lifetime_v2").fetchone()[0]
null_rates = {}
for col in cols:
    n_null = con.execute(f"SELECT COUNT(*) FROM dev_features_lifetime_v2 WHERE {col} IS NULL").fetchone()[0]
    null_rates[col] = round(100 * n_null / total, 2)

null_df = pd.DataFrame.from_dict(null_rates, orient='index', columns=['null_pct']).sort_values('null_pct', ascending=False)
display(null_df[null_df['null_pct'] > 0])

In [ ]:
numeric_cols = [
    'lifetime_activity_count', 'lifetime_activity_score_sum',
    'lifetime_learn_count', 'lifetime_evaluate_count',
    'lifetime_build_count', 'lifetime_champion_count',
    'lifetime_high_effort_count'
]
existing = [r[0] for r in con.execute("DESCRIBE dev_features_lifetime_v2").fetchall()]
numeric_cols = [c for c in numeric_cols if c in existing]

df = con.execute(f"SELECT {', '.join(numeric_cols)} FROM dev_features_lifetime_v2 WHERE lifetime_activity_count > 0").fetchdf()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].clip(upper=df[col].quantile(0.99)), bins=40, edgecolor='white')
    axes[i].set_title(col.replace('lifetime_', ''), fontsize=9)
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Lifetime feature distributions (active devs, clipped at p99)', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Windowed Features (30 / 30-90 / 90-180 days)

In [ ]:
windows = {
    '0-30d':   'dev_features_0_30d_v2',
    '30-90d':  'dev_features_30_90d_v2',
    '90-180d': 'dev_features_90_180d_v2',
}
present_tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]

for label, tbl in windows.items():
    if tbl not in present_tables:
        print(f"{tbl} not found, skipping.")
        continue
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    n_active = con.execute(f"SELECT COUNT(*) FROM {tbl} WHERE activity_count > 0").fetchone()[0]
    print(f"{label} ({tbl}): {n:,} rows, {n_active:,} active ({100*n_active/n:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (label, tbl) in zip(axes, windows.items()):
    if tbl not in present_tables:
        ax.set_visible(False)
        continue
    vals = con.execute(f"SELECT activity_count FROM {tbl} WHERE activity_count > 0").fetchdf()
    ax.hist(vals['activity_count'].clip(upper=vals['activity_count'].quantile(0.99)), bins=40, edgecolor='white')
    ax.set_title(f'activity_count ({label})')
plt.suptitle('Activity count distributions by window (active devs, p99 clip)')
plt.tight_layout()
plt.show()

## 9. Effort Level (`dev_effort_level_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_effort_level_v2").fetchdf())

el = con.execute("""
    SELECT effort_level, COUNT(*) AS n
    FROM dev_effort_level_v2
    GROUP BY effort_level ORDER BY n DESC
""").fetchdf()
display(el)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(el['effort_level'].astype(str), el['n'])
ax.set_title('Developer effort level distribution')
ax.set_xlabel('Effort level')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 10. Persona (`dev_persona_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_persona_v2").fetchdf())

In [ ]:
persona_col_candidates = ['top_persona', 'primary_persona', 'persona']
existing_cols = [r[0] for r in con.execute("DESCRIBE dev_persona_v2").fetchall()]
persona_col = next((c for c in persona_col_candidates if c in existing_cols), None)

if persona_col:
    p = con.execute(f"""
        SELECT {persona_col}, COUNT(*) AS n
        FROM dev_persona_v2
        GROUP BY {persona_col} ORDER BY n DESC
    """).fetchdf()
    display(p)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(p[persona_col].astype(str), p['n'])
    ax.set_title(f'Persona distribution ({persona_col})')
    ax.set_ylabel('Count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No top_persona/primary_persona column found — check DESCRIBE output above.")

In [ ]:
score_cols = [c for c in existing_cols if 'score' in c or 'persona' in c]
score_cols = [c for c in score_cols if c != persona_col]

if score_cols:
    df = con.execute(f"SELECT {', '.join(score_cols)} FROM dev_persona_v2").fetchdf()
    display(df[score_cols].describe())

## 11. Journey State (`dev_journey_state_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_journey_state_v2").fetchdf())

In [ ]:
js_col_candidates = ['journey_state', 'current_state', 'state']
existing_cols = [r[0] for r in con.execute("DESCRIBE dev_journey_state_v2").fetchall()]
js_col = next((c for c in js_col_candidates if c in existing_cols), None)

if js_col:
    js = con.execute(f"""
        SELECT {js_col}, COUNT(*) AS n
        FROM dev_journey_state_v2
        GROUP BY {js_col} ORDER BY n DESC
    """).fetchdf()
    display(js)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(js[js_col].astype(str), js['n'])
    ax.set_title('Journey state distribution')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No journey_state column found — check DESCRIBE output above.")

## 12. Dormancy (`dev_dormancy_status_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_dormancy_status_v2").fetchdf())

In [ ]:
dorm_col_candidates = ['dormancy_status', 'dormant', 'is_dormant', 'status']
existing_cols = [r[0] for r in con.execute("DESCRIBE dev_dormancy_status_v2").fetchall()]
dorm_col = next((c for c in dorm_col_candidates if c in existing_cols), None)

if dorm_col:
    d = con.execute(f"""
        SELECT {dorm_col}, COUNT(*) AS n
        FROM dev_dormancy_status_v2
        GROUP BY {dorm_col} ORDER BY n DESC
    """).fetchdf()
    display(d)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(d[dorm_col].astype(str), d['n'])
    ax.set_title('Dormancy status distribution')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No dormancy_status/dormant column found — check DESCRIBE output above.")

## 13. Activation (`dev_activation_v2`)

In [ ]:
display(con.execute("DESCRIBE dev_activation_v2").fetchdf())

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(CASE WHEN is_activated THEN 1 END) AS activated,
        ROUND(100.0 * COUNT(CASE WHEN is_activated THEN 1 END) / COUNT(*), 2) AS activation_rate_pct
    FROM dev_activation_v2
""").fetchdf())

In [ ]:
act_cols = [r[0] for r in con.execute("DESCRIBE dev_activation_v2").fetchall()]
days_col = next((c for c in act_cols if 'days' in c), None)

if days_col:
    vals = con.execute(f"""
        SELECT {days_col} FROM dev_activation_v2
        WHERE {days_col} IS NOT NULL AND {days_col} >= 0
    """).fetchdf()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(vals[days_col].clip(upper=vals[days_col].quantile(0.99)), bins=50, edgecolor='white')
    ax.set_title(f'{days_col} distribution (p99 clip)')
    ax.set_xlabel('Days')
    plt.tight_layout()
    plt.show()

## 14. Final Profile (`dev_profile_final_v4`)

In [ ]:
display(con.execute("DESCRIBE dev_profile_final_v4").fetchdf())

In [ ]:
cols = [r[0] for r in con.execute("DESCRIBE dev_profile_final_v4").fetchall()]
total = con.execute("SELECT COUNT(*) FROM dev_profile_final_v4").fetchone()[0]
null_rates = {}
for col in cols:
    n_null = con.execute(f"SELECT COUNT(*) FROM dev_profile_final_v4 WHERE {col} IS NULL").fetchone()[0]
    null_rates[col] = round(100 * n_null / total, 2)

null_df = pd.DataFrame.from_dict(null_rates, orient='index', columns=['null_pct']).sort_values('null_pct', ascending=False)
print(f"Total rows: {total:,}")
display(null_df[null_df['null_pct'] > 0])

In [ ]:
display(con.execute("SELECT * FROM dev_profile_final_v4 LIMIT 5").fetchdf())

In [ ]:
all_cols = [r[0] for r in con.execute("DESCRIBE dev_profile_final_v4").fetchall()]
dtypes = {r[0]: r[1] for r in con.execute("DESCRIBE dev_profile_final_v4").fetchall()}
numeric_cols = [c for c, t in dtypes.items() if any(x in t.upper() for x in ['INT', 'DOUBLE', 'FLOAT', 'DECIMAL', 'BIGINT'])]

if numeric_cols:
    df = con.execute(f"SELECT {', '.join(numeric_cols)} FROM dev_profile_final_v4").fetchdf()
    display(df.describe())

In [ ]:
if numeric_cols and len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(min(20, len(numeric_cols)), min(16, len(numeric_cols))))
    sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, linewidths=0.3,
                xticklabels=True, yticklabels=True, annot=len(numeric_cols) <= 15)
    ax.set_title('Feature correlation matrix (dev_profile_final_v4)')
    plt.tight_layout()
    plt.show()

## 15. Cross-Table Sanity Checks

In [ ]:
universe_n = con.execute("SELECT COUNT(*) FROM developer_universe_v2").fetchone()[0]

coverage_tables = [
    'dev_recency_features_v2',
    'dev_features_lifetime_v2',
    'dev_effort_level_v2',
    'dev_persona_v2',
    'dev_journey_state_v2',
    'dev_dormancy_status_v2',
    'dev_activation_v2',
    'dev_profile_final_v4',
]

present_tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
rows = []
for tbl in coverage_tables:
    if tbl not in present_tables:
        rows.append({'table': tbl, 'row_count': None, 'pct_of_universe': None})
        continue
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    rows.append({'table': tbl, 'row_count': n, 'pct_of_universe': round(100 * n / universe_n, 1)})

display(pd.DataFrame(rows))

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS profile_rows,
        COUNT(DISTINCT p.developer_id) AS unique_developers,
        COUNT(CASE WHEN a.developer_id IS NULL THEN 1 END) AS missing_in_activation,
        COUNT(CASE WHEN js.developer_id IS NULL THEN 1 END) AS missing_in_journey_state
    FROM dev_profile_final_v4 p
    LEFT JOIN dev_activation_v2 a ON p.developer_id = a.developer_id
    LEFT JOIN dev_journey_state_v2 js ON p.developer_id = js.developer_id
""").fetchdf())

## 16. Teardown

In [ ]:
con.close()
print("Connection closed.")